# Project 07: Chest X-Ray Multi-Label Pathology Classification
### Deep Learning for Thoracic Radiography & Grad-CAM Visualization

Developing multi-label classification models for clinical chest radiographs with attention heatmaps.

In [ ]:
import torch
import torch.nn as nn

class ChestXRayDenseNet(nn.Module):
    """Multi-label thoracic pathology classification backbone."""
    def __init__(self, num_pathologies: int = 4):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3), # Grayscale X-Ray
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.head = nn.Linear(64, num_pathologies)
        
    def forward(self, x):
        feats = self.backbone(x)
        flat = feats.view(feats.size(0), -1)
        return self.head(flat)

model = ChestXRayDenseNet(num_pathologies=4)
pathologies = ["Pneumonia", "Effusion", "Infiltration", "Atelectasis"]
xray_sample = torch.randn(1, 1, 224, 224)
logits = model(xray_sample)
sigmoids = torch.sigmoid(logits)[0]

print("Multi-Label Pathology Predictions:")
for path_name, prob in zip(pathologies, sigmoids):
    tag = "[POSITIVE]" if prob > 0.5 else "[NEGATIVE]"
    print(f"- {path_name:15s}: {prob.item():.2%} {tag}")